In [2]:
import numpy as np
import pandas as pd

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [5]:
df = pd.read_csv('/content/drive/MyDrive/Machine_learning/covid_toy.csv')

In [ ]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [8]:
X_train

,age,gender,fever,cough,city
12,25,Female,99.0,Strong,Kolkata
22,71,Female,98.0,Strong,Kolkata
56,71,Male,NaN,Strong,Kolkata
51,11,Female,100.0,Strong,Kolkata
40,49,Female,102.0,Mild,Delhi
...,...,...,...,...,...
85,16,Female,103.0,Mild,Bangalore
57,49,Female,99.0,Strong,Bangalore
65,69,Female,102.0,Mild,Bangalore
2,42,Male,101.0,Mild,Delhi


## 1. Aam Zindagi

# SimpleImputer supports different filling methods.

| Strategy        | Meaning                                    |
| --------------- | ------------------------------------------ |
| `mean`          | Replace missing values with column average |
| `median`        | Replace with middle value                  |
| `most_frequent` | Replace with most common value             |
| `constant`      | Replace with a fixed value                 |


In [ ]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [9]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [ ]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse=False)# Return the output as a normal dense NumPy array instead of a sparse matrix
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [ ]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

## Mentos Zindagi

In [10]:
from sklearn.compose import ColumnTransformer

In [14]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [15]:
transformer.fit_transform(X_train).shape

(80, 7)

In [16]:
transformer.transform(X_test).shape

(20, 7)

`remainder='passthrough'` is used in `ColumnTransformer` in [scikit-learn](https://scikit-learn.org/stable/?utm_source=chatgpt.com).

It means:

> "Keep all the columns that are not explicitly transformed."

---

# Why Do We Need It?

Suppose your dataset has:

| Age | Salary | Gender | City  |
| --- | ------ | ------ | ----- |
| 25  | 50000  | Male   | Delhi |

You may want to:

* Scale numerical columns
* Encode categorical columns

But what about the remaining columns?

By default, `ColumnTransformer` drops unused columns.

---

# Without `remainder='passthrough'`

Example:

```python id="1e0qmh"
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

transformer = ColumnTransformer(
    transformers=[
        ('tnf1', OneHotEncoder(), ['Gender'])
    ]
)
```

Only `Gender` will remain after transformation.

Other columns (`Age`, `Salary`, `City`) are removed.

---

# With `remainder='passthrough'`

```python id="srr02c"
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', OneHotEncoder(), ['Gender'])
    ],
    remainder='passthrough'
)
```

Now:

* `Gender` is transformed
* Remaining columns are kept unchanged

---

# Simple Example

## Dataset

```python id="0u1ndh"
import pandas as pd

df = pd.DataFrame({
    'Age': [25, 30],
    'Gender': ['Male', 'Female'],
    'Salary': [50000, 60000]
})

print(df)
```

Output:

```python id="40pk7m"
   Age  Gender  Salary
0   25    Male   50000
1   30  Female   60000
```

---

# Using ColumnTransformer

```python id="6lu5w0"
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

transformer = ColumnTransformer(
    transformers=[
        ('gender_encode', OneHotEncoder(), ['Gender'])
    ],
    remainder='passthrough'
)

result = transformer.fit_transform(df)

print(result)
```

Output:

```python id="x0prm3"
[[0.0 1.0 25 50000]
 [1.0 0.0 30 60000]]
```

Explanation:

| Encoded Gender   | Age | Salary |
| ---------------- | --- | ------ |
| Male → `[0,1]`   | 25  | 50000  |
| Female → `[1,0]` | 30  | 60000  |

`Age` and `Salary` passed through unchanged.

---

# What Happens Internally?

`ColumnTransformer` works like this:

1. Apply transformation to selected columns
2. Ignore or keep remaining columns
3. Combine everything into one output matrix

---

# Other Options for `remainder`

| Value              | Meaning                            |
| ------------------ | ---------------------------------- |
| `'drop'`           | Remove remaining columns (default) |
| `'passthrough'`    | Keep remaining columns             |
| transformer object | Apply another transformer          |

---

# Example with `drop`

```python id="jlwm7o"
remainder='drop'
```

Only transformed columns remain.

---

# Example with Another Transformer

```python id="2n3p6j"
from sklearn.preprocessing import StandardScaler

remainder=StandardScaler()
```

Now all remaining columns will also be scaled.

---

# Real-Life Usage

Very common in ML pipelines:

* Encode categorical columns
* Scale numerical columns
* Keep untouched columns

Example:

```python id="6d6j69"
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age', 'Salary']),
        ('cat', OneHotEncoder(), ['Gender'])
    ],
    remainder='passthrough'
)
```

---

# Important Point

The order of columns changes after transformation.

Transformed columns usually appear first, followed by passed-through columns.

---

# Summary

| Parameter                 | Meaning                     |
| ------------------------- | --------------------------- |
| `remainder='drop'`        | Remove unused columns       |
| `remainder='passthrough'` | Keep unused columns         |
| Used in                   | `ColumnTransformer`         |
| Main purpose              | Preserve remaining features |
